# Parámetros guardados y activos por token en los cien modelos en tendencia del Hub

Cuaderno de lectura de la medición `hf-activos/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Respalda el artículo [cinco-de-cada-cien](https://manpla.net/posts/cinco-de-cada-cien/). Carga el fichero de al lado —o lo descarga del repositorio si se ejecuta fuera de él—, muestra la ficha de procedencia y dibuja una figura con matplotlib a secas. Solo lee; no vuelve a tomar la instantánea: para eso está `generar.py`.

*Reading notebook for this measurement: loads the file next to it, prints the provenance record and draws one figure. Column names are in Spanish; `GLOSARIO.md` gives the English form.*

In [ ]:
import io, json, urllib.request
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/hf-activos/"

def leer(nombre, **kw):
    """el fichero de al lado si existe; si no, el del repositorio."""
    p = Path(nombre)
    if p.exists():
        return pd.read_csv(p, **kw)
    return pd.read_csv(RAW + nombre, **kw)

def texto(nombre):
    p = Path(nombre)
    if p.exists():
        return p.read_text(encoding="utf-8")
    with urllib.request.urlopen(RAW + nombre, timeout=30) as r:
        return r.read().decode("utf-8")


## Ficha de procedencia

In [ ]:
print(texto("INSTANTANEA.md"))

## El dato

In [ ]:
a = leer("activos.csv")
print(a.tipo.value_counts().to_dict())
a[a.tipo == "moe"].sort_values("fraccion_activa")[["modelo", "params_total", "params_activos", "fraccion_activa", "expertos_por_capa", "activos_por_token"]]

## Una figura

In [ ]:
m = a[a.tipo == "moe"].sort_values("fraccion_activa"); d = a[a.tipo == "denso"]
fig, (x, y) = plt.subplots(1, 2, figsize=(12, 4.5))
x.scatter(d.params_total, d.params_activos, s=18, label="densos"); x.scatter(m.params_total, m.params_activos, s=30, color="C3", label="mezcla de expertos")
x.set_xscale("log"); x.set_yscale("log"); x.set_xlabel("parámetros guardados"); x.set_ylabel("activos por token"); x.legend()
y.barh(m.modelo.str.split("/").str[-1], 100 * m.fraccion_activa, color="C3"); y.set_xlabel("% activo por token"); plt.tight_layout()